# **Car Price Analysis**

## Objectives




## Inputs

Data set content: For this project the data source Kaggle and there are just 205 cars.  The dataset tells us details about the car, from its size, engine specification, fuel, power, aspiration (turbo) efficiency and price. Limitations is we’re not told where in the US its sold, nor by who.  We can assume they’re all new cars without mileage and randomly sourced from various states around the US.   This potentially limits the reliability of the conclusions for example location bias due to climate or state wealth.


## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 


## Additional Comments

Business requirements and goals: They want to finds positive coefficients in any of the data they’ve obtained to tell them which features of cars American buyers are willing to pay more for.  So, aim is to identify which features are worth investing in for their export market and to design vehicles that match American preferences to boost sales.


## What you are looking for?

 Questions. Which answer is using Analytics to support Making an evidence based business decision.

 ## Project Plan ##
 
 What is the plan to reach the business requirements and answer the questions?

---

In [1]:
import numpy as np
import pandas as pd

In [53]:
##New code
##this code loads the csv directly from data sets added to Kaggle.
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "CarPrice_Assignment.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "hellbuoy/car-price-prediction",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

C:\Users\manager\AppData\Local\Temp\ipykernel_15436\23357354.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:    car_ID  symboling                   CarName fueltype aspiration doornumber  \
0       1          3        alfa-romero giulia      gas        std        two   
1       2          3       alfa-romero stelvio      gas        std        two   
2       3          1  alfa-romero Quadrifoglio      gas        std        two   
3       4          2               audi 100 ls      gas        std       four   
4       5          2                audi 100ls      gas        std       four   

       carbody drivewheel enginelocation  wheelbase  ...  enginesize  \
0  convertible        rwd          front       88.6  ...         130   
1  convertible        rwd          front       88.6  ...         130   
2    hatchback        rwd          front       94.5  ...         152   
3        sedan        fwd          front       99.8  ...         109   
4        sedan        4wd          front       99.4  ...         136   

   fuelsystem  boreratio  stroke compressionratio horsepower  p

## Understanding the data
Explore, get familiar with the structure. Take steps to clean. Follow basic cleaning steps as planned.
Data loss or data inconsistancy can be misleading if the scale of the data is small. Error can change a result. As the set it small its worth doing.

In [57]:
#df.shape # 205 rows with 26
#df.info() ##checking for size, nulls and types.  No issues.  No null values.
df.nunique()  # The carname field isn't too useful. The model would be useful to group by if it was seperated cleanly.  Some data living in the name need checking against other fields.
data_cleaned = df.copy() 
duplicate_count = data_cleaned.duplicated().sum()# No duplicate to sort out
missing_data = data_cleaned.isnull().sum() # no null
# Convert all fields to lowercase
df.columns = df.columns.str.lower()
df.columns



Index(['car_id', 'symboling', 'carname', 'fueltype', 'aspiration',
       'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase',
       'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype',
       'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')

In [ ]:
df["carname"]
# Splits into a list per row, then extracts index 0 as a string for the car's Make to its own field.
df["make"] = df["carname"].str.split(' ').str[0]
df["make"]
# Displays an array of all distinct make values
print(df["make"].unique())
## wrong models. vw>volswagen

['alfa-romero' 'audi' 'bmw' 'chevrolet' 'dodge' 'honda' 'isuzu' 'jaguar'
 'maxda' 'mazda' 'buick' 'mercury' 'mitsubishi' 'Nissan' 'nissan'
 'peugeot' 'plymouth' 'porsche' 'porcshce' 'renault' 'saab' 'subaru'
 'toyota' 'toyouta' 'vokswagen' 'volkswagen' 'vw' 'volvo']


In [43]:
# Create the dictionary of error I found.
make_replacements = {
    'vw': 'volkswagen',
    'vokswagen': 'volkswagen',
    'toyouta': 'toyota',
    'maxda': 'mazda',
    'porcshce': 'porsche',
    'Nissan': 'nissan',          
    'alfa-romero': 'alfa-romeo'    
}

df['make'] = df['make'].replace(make_replacements)
#Standardising non-capitalisation in the whole set.
df['make'] = df['make'].str.lower()
df
#observer whole dataset to spot things in car name we still can use before we drop the column.
#(Auto)- reference to transmission which we don't have.      (diesel) - reference to fueltype which needs looking at to confirm.
#(turbo) - reference to aspiration which we need to confirm,
# Takes time. using AI to help find what wlse is hidden in carname. Copy and paste all column and carname field to gemini.

,car_id,symboling,carname,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,make
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,alfa-romeo
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,alfa-romeo
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,alfa-romeo
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0,audi
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0,audi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0,volvo
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0,volvo
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0,volvo
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,idi,3.01,3.40,23.0,106,4800,26,27,22470.0,volvo


In [44]:
# 1. Fix 'fueltype'
# Check if carname mentions 'diesel' and current fueltype is not already 'diesel'
diesel_mask = df['carname'].str.contains('diesel', case=False, na=False)
df.loc[diesel_mask & (df['fueltype'] != 'diesel'), 'fueltype'] = 'diesel'

# 2. Fix 'aspiration'
# Check if carname mentions 'turbo' and current aspiration is not already 'turbo'
turbo_mask = df['carname'].str.contains('turbo', case=False, na=False)
df.loc[turbo_mask & (df['aspiration'] != 'turbo'), 'aspiration'] = 'turbo'

# 3. Fix 'carbody'
# Mapping parenthetical wagon flags like '(sw)' to 'wagon' only if not already set
wagon_mask = df['carname'].str.contains(r'\(sw\)', case=False, na=False)
df.loc[wagon_mask & (df['carbody'] != 'wagon'), 'carbody'] = 'wagon'



##Rejected AI suggestions.
## Mapping coupe indicators like 'coupe' in carname
#coupe_mask = df['carname'].str.contains('coupe', case=False, na=False)
#df.loc[coupe_mask & (df['carbody'] != 'coupe'), 'carbody'] = 'coupe'
#After review Take a look at the row buick regal sport coupe (turbo). 
# In automotive terms, manufacturers often sold trim levels called "Sport Coupe" on 2-door
#  hardtops or hatchbacks, while the database schema might classify the official body style as a 
#  hatchback or hardtop. Forcing carbody to 'coupe' based solely on a marketing trim name in 
#  carname overwrites the original, more precise carbody classification.
# 4. Extract and create 'transmission' column (New Field)
##These change suggestion were skipped.
# 1. Deterministic Extraction (High Confidence)
# These are exact parenthetical patterns where we are 
# 100% sure what the text means:(sw) $\rightarrow$ carbody = 'wagon'(auto) $\rightarrow$ transmission = 'automatic'(turbo) $\rightarrow$ aspiration = 'turbo'(diesel) $\rightarrow$ fueltype = 'diesel'

In [ ]:
#check for invalid text based field ising value_counts() function.
for field in df[["fueltype", "aspiration", "doornumber", "carbody", "drivewheel", "enginelocation", "enginetype"]]:
    print(df[field].value_counts(), end='\n|\n')

#Some are rare but all valid entries. Rare types can be grouped together as 'rare' to reduce noise.    